### 네이버 뉴스 기반 QA 챗봇

In [1]:
!uv --version

uv 0.12.17 (635500036 2026-09-18 x86_64-pc-windows-msvc)


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_teddynote import logging

logging.langsmith("test0925")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0925


In [ ]:
# !uv add bs4

In [9]:
import bs4
from langchain_classic import hub
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [10]:
loader = WebBaseLoader(
    web_paths=("https://n.news.naver.com/article/437/0000378416",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "div",
            attrs={"class": ["newsct_article_article_body", "media_end_head_title"]},
        )
    ),
)

docs = loader.load()
print(f"문서의 수: {len(docs)}")
docs

문서의 수: 1


[Document(metadata={'source': 'https://n.news.naver.com/article/437/0000378416'}, page_content="\n출산 직원에게 '1억원' 쏜다…회사의 파격적 저출생 정책\n")]

In [11]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

splits = text_splitter.split_documents(docs)
len(splits)

1

In [12]:
vectorstore = FAISS.from_documents(documents=splits, embedding=OpenAIEmbeddings())

retriever = vectorstore.as_retriever()

In [13]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """당신은 질문-답변을 수행하는 AI어시스턴트입니다. 당신의 임무는 주어진 문맥에서 주어진 질문에 답하는 것입니다.
    검색된 다음 문맥을 사용하여 질문에 답하세요. 만약, 주어진 문맥에서 답을 찾을 수 없다면, 답을 모른다면 '주어진 정보에서 질문에 대한 정보를 찾을 수 없습니다' 라고 답하세요.
    한글로 답변해 주세요. 단, 기술적인 용어나 이름은 번역하지 않고 그대로 사용해 주세요.
    
    #Question:
    {question}
    
    #Context:
    {context}
    
    #Answer:"""
)

In [14]:
# prompt = hub.pull("teddynote/rag-prompt-korean")
# prompt

In [18]:
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [19]:
from langchain_teddynote.messages import stream_response

In [20]:
answer = rag_chain.stream("부영그룹의 출산 장려 정책에 대해 설명해주세요.")
stream_response(answer)

부영그룹은 출산 직원에게 1억원을 지원하는 파격적인 출산 장려 정책을 시행하고 있습니다. 이 정책은 저출생 문제를 해결하기 위한 노력의 일환으로, 출산을 장려하기 위해 마련된 것입니다.

In [22]:
answer = rag_chain.stream("부영그룹은 출산 직원에게 얼마의 지원을 제공하나요?")
stream_response(answer)

부영그룹은 출산 직원에게 1억원의 지원을 제공합니다.

In [23]:
answer = rag_chain.stream("정부의 저출생 대책을 bullet points 형식으로 작성해 주세요.")
stream_response(answer)

주어진 정보에서 질문에 대한 정보를 찾을 수 없습니다.